# 02. Python Streamlit 사용 예제

농업 데이터를 예로 들어 **Streamlit**으로 인터랙티브 대시보드를 만드는 방법을 학습합니다.

| 항목 | 설명 |
|------|------|
| Streamlit이란? | Python만으로 웹 대시보드를 빠르게 만드는 라이브러리 |
| 실행 방식 | `.py` 파일을 `streamlit run`으로 실행 |
| Jupyter와의 차이 | 노트북 안에서 직접 렌더링되지 않고, 별도 웹 서버로 동작 |
| 강점 | 위젯(선택박스, 슬라이더 등) + 차트 + 표를 짧은 코드로 구성 |
| 이 예제의 목표 | 작물별 수확량 대시보드 앱 파일을 만들고 실행하기 |

## 0. 라이브러리 설치 (필요 시)

아래 셀은 한 번만 실행하면 됩니다.

In [1]:
# 사용법: Streamlit 앱 실행에 필요한 패키지를 설치합니다.
# - streamlit : 웹 대시보드 프레임워크
# - pandas    : 표 형태 데이터 처리
# - plotly    : 인터랙티브 차트 (Streamlit과 잘 맞음)
# - numpy     : 숫자 연산 / 샘플 데이터 생성
%pip install streamlit pandas plotly numpy -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Streamlit 기본 개념

Streamlit 앱은 **위에서 아래로 한 번 실행되는 스크립트**입니다.  
사용자가 위젯(예: 선택박스)을 바꾸면 스크립트가 **처음부터 다시 실행**됩니다.

자주 쓰는 함수:

| 함수 | 용도 |
|------|------|
| `st.title()` / `st.header()` | 제목 / 소제목 |
| `st.write()` / `st.markdown()` | 텍스트·마크다운 출력 |
| `st.dataframe()` | 데이터프레임 표 표시 |
| `st.selectbox()` / `st.slider()` | 사용자 입력 위젯 |
| `st.plotly_chart()` | Plotly 차트 표시 |
| `st.sidebar` | 왼쪽 사이드바 영역 |
| `st.columns()` | 화면을 여러 열로 분할 |
| `st.metric()` | KPI 숫자 카드 |

## 2. 샘플 농업 데이터 미리보기 (노트북에서 확인)

앱에 넣을 데이터를 먼저 노트북에서 확인합니다.  
실제 대시보드 화면은 이후 `.py` 파일을 `streamlit run`으로 실행해야 보입니다.

In [2]:
import numpy as np
import pandas as pd

# 사용법: 월별·작물별 가상 농업 데이터를 만듭니다.
# 실제 프로젝트에서는 CSV/DB에서 읽어오면 됩니다. (예: pd.read_csv("data.csv"))
rng = np.random.default_rng(42)
months = [f"{m}월" for m in range(1, 13)]
crops = ["벼", "옥수수", "감자", "토마토"]

rows = []
for crop in crops:
    base = {"벼": 40, "옥수수": 55, "감자": 30, "토마토": 25}[crop]
    for i, month in enumerate(months, start=1):
        # 사용법: 계절성(sin) + 노이즈로 현실감 있는 수확량/기온/강수량을 생성
        harvest = base + 20 * np.sin((i - 3) / 12 * 2 * np.pi) + rng.normal(0, 3)
        temp = 12 + 15 * np.sin((i - 3) / 12 * 2 * np.pi) + rng.normal(0, 1)
        rain = 60 + 40 * np.sin((i - 1) / 12 * 2 * np.pi) + rng.normal(0, 8)
        rows.append(
            {
                "월": month,
                "월번호": i,
                "작물": crop,
                "수확량_톤": round(max(harvest, 5), 1),
                "기온_C": round(temp, 1),
                "강수량_mm": round(max(rain, 5), 1),
            }
        )

df = pd.DataFrame(rows)
df.head()  # 사용법: 상위 5행을 미리 확인

,월,월번호,작물,수확량_톤,기온_C,강수량_mm
0,1월,1,벼,23.6,-2.0,66.0
1,2월,2,벼,32.8,2.5,69.6
2,3월,3,벼,40.4,11.7,94.5
3,4월,4,벼,47.4,20.4,106.2
4,5월,5,벼,57.5,26.1,98.4


In [3]:
# 사용법: 작물별 요약 통계를 확인합니다.
# groupby + agg는 Streamlit에서 KPI(평균 수확량 등)를 만들 때 자주 씁니다.
summary = (
    df.groupby("작물", as_index=False)
    .agg(평균수확량=("수확량_톤", "mean"), 총수확량=("수확량_톤", "sum"), 평균기온=("기온_C", "mean"))
    .round(1)
)
summary

,작물,평균수확량,총수확량,평균기온
0,감자,29.1,349.0,11.7
1,벼,40.3,483.6,12.0
2,옥수수,55.2,662.5,11.7
3,토마토,25.9,311.1,11.9


## 3. Streamlit 핵심 API 미니 예제 (참고용 코드)

> 아래 코드는 **개념 설명용**입니다. Jupyter 셀에서 실행해도 웹 UI가 뜨지 않습니다.  
> 실제 화면은 **4절에서 생성한 `.py` 파일**을 실행해야 합니다.

In [4]:
# ============================================================
# [참고용] Streamlit 핵심 위젯 / 출력 함수 사용법
# - 이 셀은 Jupyter에서 실행해도 대시보드가 열리지 않습니다.
# - 문법을 익히기 위한 예시이므로, 실제 앱은 4절의 .py 파일을 사용하세요.
# ============================================================

streamlit_api_cheatsheet = r'''
import streamlit as st
import pandas as pd
import plotly.express as px

# 사용법: 페이지 설정은 스크립트에서 가장 먼저 한 번만 호출
st.set_page_config(page_title="농업 대시보드", layout="wide")

# 사용법: 제목 / 설명 텍스트
st.title("농업 데이터 대시보드")
st.markdown("작물별 수확량과 기상 지표를 살펴봅니다.")

# 사용법: 사이드바 위젯 — 왼쪽 패널에 필터를 두면 본문이 깔끔해집니다
crop = st.sidebar.selectbox("작물 선택", ["벼", "옥수수", "감자", "토마토"])
month_range = st.sidebar.slider("월 범위", 1, 12, (1, 12))

# 사용법: KPI 카드 (현재값, 전월 대비 등)
col1, col2, col3 = st.columns(3)
col1.metric("평균 수확량", "42.5 톤", delta="+3.1")
col2.metric("평균 기온", "18.2 °C", delta="-0.4")
col3.metric("평균 강수량", "72 mm", delta="+5")

# 사용법: 표 표시 (interactive dataframe)
st.dataframe(pd.DataFrame({"월": ["1월", "2월"], "수확량_톤": [12, 15]}))

# 사용법: Plotly 차트를 Streamlit에 넣기
fig = px.line(x=["1월", "2월"], y=[12, 15], title="수확량 추이")
st.plotly_chart(fig, use_container_width=True)
'''

print("아래는 Streamlit API 치트시트(문자열)입니다. 실제 실행은 4절 .py 파일을 사용하세요.\n")
print(streamlit_api_cheatsheet)

아래는 Streamlit API 치트시트(문자열)입니다. 실제 실행은 4절 .py 파일을 사용하세요.


import streamlit as st
import pandas as pd
import plotly.express as px

# 사용법: 페이지 설정은 스크립트에서 가장 먼저 한 번만 호출
st.set_page_config(page_title="농업 대시보드", layout="wide")

# 사용법: 제목 / 설명 텍스트
st.title("농업 데이터 대시보드")
st.markdown("작물별 수확량과 기상 지표를 살펴봅니다.")

# 사용법: 사이드바 위젯 — 왼쪽 패널에 필터를 두면 본문이 깔끔해집니다
crop = st.sidebar.selectbox("작물 선택", ["벼", "옥수수", "감자", "토마토"])
month_range = st.sidebar.slider("월 범위", 1, 12, (1, 12))

# 사용법: KPI 카드 (현재값, 전월 대비 등)
col1, col2, col3 = st.columns(3)
col1.metric("평균 수확량", "42.5 톤", delta="+3.1")
col2.metric("평균 기온", "18.2 °C", delta="-0.4")
col3.metric("평균 강수량", "72 mm", delta="+5")

# 사용법: 표 표시 (interactive dataframe)
st.dataframe(pd.DataFrame({"월": ["1월", "2월"], "수확량_톤": [12, 15]}))

# 사용법: Plotly 차트를 Streamlit에 넣기
fig = px.line(x=["1월", "2월"], y=[12, 15], title="수확량 추이")
st.plotly_chart(fig, use_container_width=True)



## 4. 완성 예제 앱 파일 생성

아래 셀을 실행하면 같은 폴더에 `streamlit_example.py` 파일이 만들어집니다.  
코드 안의 **주석**에 각 함수 사용법을 적어 두었습니다.

In [5]:
from pathlib import Path

# 사용법: 노트북과 같은 폴더에 Streamlit 앱(.py)을 저장합니다.
app_path = Path("streamlit_example.py")

app_code = r'''"""
농업 데이터 시각화 대시보드 (Streamlit 예제)

실행 방법
--------
1) 터미널에서 이 파일이 있는 폴더로 이동
2) 아래 명령 실행:
       streamlit run sreamlit_example.py
3) 브라우저가 자동으로 열리면 대시보드를 확인

중지 방법
--------
- 터미널에서 Ctrl + C
"""

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st


# ------------------------------------------------------------
# 사용법: set_page_config()
# - 페이지 제목, 아이콘, 레이아웃(wide/centered)을 설정합니다.
# - 반드시 다른 st.* 호출보다 먼저, 그리고 한 번만 호출하세요.
# ------------------------------------------------------------
st.set_page_config(
    page_title="농업 데이터 대시보드",
    page_icon="🌾",
    layout="wide",
)


@st.cache_data
def load_agriculture_data() -> pd.DataFrame:
    """가상 농업 데이터를 생성합니다.

    사용법:
        - @st.cache_data : 같은 입력이면 결과를 캐시해 재계산을 줄입니다.
        - 실제 업무에서는 아래 생성 로직을 pd.read_csv / DB 조회로 바꾸면 됩니다.
    """
    rng = np.random.default_rng(42)
    months = [f"{m}월" for m in range(1, 13)]
    crops = ["벼", "옥수수", "감자", "토마토"]

    rows = []
    for crop in crops:
        base = {"벼": 40, "옥수수": 55, "감자": 30, "토마토": 25}[crop]
        for i, month in enumerate(months, start=1):
            harvest = base + 20 * np.sin((i - 3) / 12 * 2 * np.pi) + rng.normal(0, 3)
            temp = 12 + 15 * np.sin((i - 3) / 12 * 2 * np.pi) + rng.normal(0, 1)
            rain = 60 + 40 * np.sin((i - 1) / 12 * 2 * np.pi) + rng.normal(0, 8)
            rows.append(
                {
                    "월": month,
                    "월번호": i,
                    "작물": crop,
                    "수확량_톤": round(max(harvest, 5), 1),
                    "기온_C": round(temp, 1),
                    "강수량_mm": round(max(rain, 5), 1),
                }
            )
    return pd.DataFrame(rows)


# 사용법: 데이터 로드 (캐시됨)
df = load_agriculture_data()

# ------------------------------------------------------------
# 사용법: title / caption
# - st.title()  : 페이지 최상단 큰 제목
# - st.caption(): 보조 설명(작은 회색 텍스트)
# ------------------------------------------------------------
st.title("농업 데이터 시각화 대시보드")
st.caption("Streamlit + Pandas + Plotly 예제 | 작물·월 필터로 수확량과 기상을 탐색합니다.")

# ------------------------------------------------------------
# 사용법: sidebar 위젯
# - st.sidebar.selectbox(label, options) : 드롭다운 선택
# - st.sidebar.multiselect(...)          : 다중 선택
# - st.sidebar.slider(min, max, value)   : 범위/숫자 슬라이더
# 반환값 = 사용자가 현재 선택한 값 (위젯이 바뀌면 스크립트가 다시 실행됨)
# ------------------------------------------------------------
st.sidebar.header("필터")

selected_crops = st.sidebar.multiselect(
    "작물 선택",
    options=sorted(df["작물"].unique()),
    default=sorted(df["작물"].unique()),  # 사용법: default로 초기 선택값 지정
    help="비교할 작물을 하나 이상 선택하세요.",  # 사용법: help는 ? 아이콘 툴팁
)

month_start, month_end = st.sidebar.slider(
    "월 범위",
    min_value=1,
    max_value=12,
    value=(1, 12),  # 사용법: 튜플이면 범위 슬라이더가 됩니다
)

chart_type = st.sidebar.radio(
    "차트 종류",
    options=["선 그래프", "막대 그래프", "산점도"],
    index=0,  # 사용법: 기본으로 첫 번째 옵션 선택
)

show_raw = st.sidebar.checkbox("원본 데이터 보기", value=False)

# 사용법: 선택이 비어 있으면 안내 메시지를 보여주고 중단
if not selected_crops:
    st.warning("사이드바에서 작물을 하나 이상 선택해 주세요.")
    st.stop()  # 사용법: st.stop() 이후 코드는 실행되지 않습니다

# 사용법: 사이드바 선택값으로 데이터 필터링
filtered = df[
    (df["작물"].isin(selected_crops))
    & (df["월번호"].between(month_start, month_end))
].copy()

# ------------------------------------------------------------
# 사용법: metric + columns
# - st.columns(n) : 화면을 n개 열로 나눕니다
# - st.metric(label, value, delta=...) : KPI 카드
# ------------------------------------------------------------
st.subheader("핵심 지표 (KPI)")
k1, k2, k3, k4 = st.columns(4)

avg_harvest = filtered["수확량_톤"].mean()
total_harvest = filtered["수확량_톤"].sum()
avg_temp = filtered["기온_C"].mean()
avg_rain = filtered["강수량_mm"].mean()

k1.metric("평균 수확량", f"{avg_harvest:.1f} 톤")
k2.metric("총 수확량", f"{total_harvest:.1f} 톤")
k3.metric("평균 기온", f"{avg_temp:.1f} °C")
k4.metric("평균 강수량", f"{avg_rain:.1f} mm")

st.divider()  # 사용법: 시각적 구분선

# ------------------------------------------------------------
# 사용법: Plotly 차트 + st.plotly_chart
# - use_container_width=True 이면 컨테이너 너비에 맞춰 늘어납니다
# ------------------------------------------------------------
left, right = st.columns((2, 1))

with left:
    st.subheader("월별 수확량")

    if chart_type == "선 그래프":
        # 사용법: px.line — 시간 추이(선) 시각화
        fig = px.line(
            filtered,
            x="월",
            y="수확량_톤",
            color="작물",
            markers=True,
            title="작물별 월간 수확량 추이",
        )
    elif chart_type == "막대 그래프":
        # 사용법: px.bar — 범주 비교(막대) 시각화
        fig = px.bar(
            filtered,
            x="월",
            y="수확량_톤",
            color="작물",
            barmode="group",
            title="작물별 월간 수확량 비교",
        )
    else:
        # 사용법: px.scatter — 두 변수 관계(산점도)
        fig = px.scatter(
            filtered,
            x="기온_C",
            y="수확량_톤",
            color="작물",
            size="강수량_mm",
            hover_data=["월"],
            title="기온 vs 수확량 (점 크기 = 강수량)",
        )

    st.plotly_chart(fig, use_container_width=True)

with right:
    st.subheader("작물별 합계")
    by_crop = (
        filtered.groupby("작물", as_index=False)["수확량_톤"]
        .sum()
        .sort_values("수확량_톤", ascending=False)
    )
    fig_pie = px.pie(
        by_crop,
        names="작물",
        values="수확량_톤",
        title="선택 기간 수확량 비중",
        hole=0.35,  # 사용법: hole > 0 이면 도넛 차트
    )
    st.plotly_chart(fig_pie, use_container_width=True)

# ------------------------------------------------------------
# 사용법: dataframe / download_button
# - st.dataframe : 스크롤·정렬 가능한 표
# - st.download_button : CSV 등 파일 다운로드 버튼
# ------------------------------------------------------------
st.subheader("요약 테이블")
summary = (
    filtered.groupby("작물", as_index=False)
    .agg(
        평균수확량=("수확량_톤", "mean"),
        총수확량=("수확량_톤", "sum"),
        평균기온=("기온_C", "mean"),
        평균강수량=("강수량_mm", "mean"),
    )
    .round(1)
)
st.dataframe(summary, use_container_width=True)

csv_bytes = filtered.to_csv(index=False).encode("utf-8-sig")  # 사용법: 엑셀 한글용 utf-8-sig
st.download_button(
    label="필터된 데이터 CSV 다운로드",
    data=csv_bytes,
    file_name="agriculture_filtered.csv",
    mime="text/csv",
)

if show_raw:
    st.subheader("원본(필터) 데이터")
    st.dataframe(filtered, use_container_width=True)

# 사용법: expander — 접었다 펼 수 있는 도움말 영역
with st.expander("Streamlit 사용 팁"):
    st.markdown(
        """
        - 위젯 값이 바뀌면 **스크립트 전체가 다시 실행**됩니다.
        - 무거운 데이터 로딩은 `@st.cache_data`로 감싸세요.
        - 레이아웃은 `st.sidebar`, `st.columns`, `st.tabs`로 구성합니다.
        - 배포는 Streamlit Community Cloud 또는 Docker/서버에 `streamlit run`으로 가능합니다.
        """
    )
'''

app_path.write_text(app_code, encoding="utf-8")
print(f"앱 파일 생성 완료: {app_path.resolve()}")
print("다음 셀(또는 터미널)에서 streamlit run 으로 실행하세요.")

앱 파일 생성 완료: C:\MyCursorLab\03_농업 데이터 시각화 대시보드 만들기\streamlit_example.py
다음 셀(또는 터미널)에서 streamlit run 으로 실행하세요.


## 5. 앱 실행 방법

### 방법 A — 터미널에서 실행 (권장)

```bash
streamlit run sreamlit_example.py
```

실행 후 브라우저에서 보통 `http://localhost:8501` 주소로 대시보드가 열립니다.

### 방법 B — 노트북에서 백그라운드 실행

아래 셀을 실행하면 Streamlit 서버를 백그라운드로 띄웁니다.  
(이미 실행 중이면 포트를 바꾸거나 기존 프로세스를 종료하세요.)

In [6]:
import subprocess
import sys
from pathlib import Path

# 사용법:
# 1) 먼저 4절 셀을 실행해 sreamlit_example.py 를 생성합니다.
# 2) 이 셀을 실행하면 Streamlit 서버가 백그라운드로 시작됩니다.
# 3) 브라우저에서 http://localhost:8501 을 엽니다.
# 4) 중지하려면 터미널/작업관리자에서 해당 프로세스를 종료하거나,
#    아래 stop 셀(선택)을 사용하세요.

app_file = Path("sreamlit_example.py")
if not app_file.exists():
    raise FileNotFoundError("sreamlit_example.py 가 없습니다. 먼저 4절 셀을 실행하세요.")

# 사용법: streamlit run <파일> --server.headless true
# - headless=true : 서버만 띄우고 OS 기본 브라우저 자동 실행을 줄입니다
proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(app_file),
        "--server.headless",
        "true",
        "--server.port",
        "8501",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(f"Streamlit 시작 (PID={proc.pid})")
print("브라우저에서 http://localhost:8501 을 여세요.")
print("중지: 다음 셀에서 proc.terminate() 실행 또는 터미널에서 Ctrl+C")

FileNotFoundError: sreamlit_example.py 가 없습니다. 먼저 4절 셀을 실행하세요.

In [ ]:
# 사용법: 위에서 띄운 Streamlit 프로세스를 종료합니다.
# (5절 실행 셀을 먼저 실행한 같은 커널에서만 proc 변수를 사용할 수 있습니다.)
try:
    proc.terminate()
    print("Streamlit 프로세스를 종료했습니다.")
except NameError:
    print("종료할 proc 변수가 없습니다. 이미 종료되었거나 다른 커널일 수 있습니다.")

### 한눈에 보는 실행 순서

1. `0`절 패키지 설치  
2. `4`절 셀 실행 → `streamlit_example.py` 생성  
3. 터미널: `streamlit run streamlit_example.py`  
4. 브라우저에서 대시보드 확인